# The Text Editor Tool (`005_text_editor_tool.ipynb`)

En este notebook practicamos el uso de la herramienta integrada **Text Editor Tool** de Claude (`str_replace_editor`), que permite visualizar, crear, insertar, reemplazar y deshacer ediciones en archivos y directorios.

In [20]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic
import os
import shutil
from pathlib import Path

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"  # 3-7-latest ya no existe (404); 4.x usa otro schema de text editor

In [22]:
# Helper functions
def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if hasattr(message, "content") else message
    }
    messages.append(user_message)

def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if hasattr(message, "content") else message
    }
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 2000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }
    if tools:
        params["tools"] = tools
    if system:
        params["system"] = system

    response = client.messages.create(**params)
    return response

def text_from_message(message):
    return "\n".join(
        [block.text for block in message.content if block.type == "text"]
    )

In [25]:
# Implementation of the TextEditorTool
class TextEditorTool:
    def __init__(self, base_dir=".", backup_dir=".editor_backups"):
        self.base_dir = Path(base_dir).resolve()
        self.backup_dir = self.base_dir / backup_dir
        self.backup_dir.mkdir(parents=True, exist_ok=True)

    def _resolve_path(self, path_str):
        target = (self.base_dir / path_str).resolve()
        if not str(target).startswith(str(self.base_dir)):
            raise ValueError(f"Access denied: {path_str} is outside base directory")
        return target

    def _backup_file(self, file_path):
        if file_path.exists() and file_path.is_file():
            backup_path = self.backup_dir / f"{file_path.name}.bak"
            shutil.copy2(file_path, backup_path)

    def view(self, path, view_range=None):
        target = self._resolve_path(path)
        if not target.exists():
            return f"Error: Path '{path}' does not exist."
        if target.is_dir():
            items = [f"{p.name}{'/' if p.is_dir() else ''}" for p in sorted(target.iterdir()) if not p.name.startswith(".")]
            return "\n".join(items) if items else "(empty directory)"
        lines = target.read_text(encoding="utf-8").splitlines()
        if view_range:
            start, end = view_range
            start_idx = max(0, start - 1)
            end_idx = len(lines) if end == -1 else min(len(lines), end)
            numbered = [f"{i+1}: {line}" for i, line in enumerate(lines[start_idx:end_idx], start=start_idx)]
            return "\n".join(numbered)
        return "\n".join([f"{i+1}: {line}" for i, line in enumerate(lines)])

    def str_replace(self, path, old_str, new_str):
        target = self._resolve_path(path)
        if not target.exists() or not target.is_file():
            return f"Error: File '{path}' does not exist."
        content = target.read_text(encoding="utf-8")
        count = content.count(old_str)
        if count == 0:
            return f"Error: old_str not found in '{path}'."
        if count > 1:
            return f"Error: old_str appears {count} times in '{path}'. Provide more context to make it unique."
        self._backup_file(target)
        updated = content.replace(old_str, new_str, 1)
        target.write_text(updated, encoding="utf-8")
        return f"Successfully replaced text in '{path}'."

    def create(self, path, file_text):
        target = self._resolve_path(path)
        if target.exists():
            return f"Error: File '{path}' already exists."
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(file_text, encoding="utf-8")
        return f"Successfully created '{path}'."

    def insert(self, path, insert_line, new_str):
        target = self._resolve_path(path)
        if not target.exists() or not target.is_file():
            return f"Error: File '{path}' does not exist."
        self._backup_file(target)
        lines = target.read_text(encoding="utf-8").splitlines()
        idx = max(0, min(len(lines), insert_line))
        lines.insert(idx, new_str)
        target.write_text("\n".join(lines) + "\n", encoding="utf-8")
        return f"Successfully inserted text at line {insert_line} in '{path}'."

    def undo_edit(self, path):
        target = self._resolve_path(path)
        backup_path = self.backup_dir / f"{target.name}.bak"
        if not backup_path.exists():
            return f"Error: No backup found for '{path}'."
        shutil.copy2(backup_path, target)
        return f"Successfully reverted last edit to '{path}'."

editor_tool = TextEditorTool()

In [26]:
# Process Tool Call Requests
def run_tool(tool_name, tool_input):
    if tool_name in ("str_replace_editor", "str_replace_based_edit_tool"):
        command = tool_input.get("command")
        path = tool_input.get("path")
        if command == "view":
            return editor_tool.view(path, tool_input.get("view_range"))
        elif command == "str_replace":
            return editor_tool.str_replace(path, tool_input["old_str"], tool_input["new_str"])
        elif command == "create":
            return editor_tool.create(path, tool_input["file_text"])
        elif command == "insert":
            return editor_tool.insert(path, tool_input["insert_line"], tool_input["new_str"])
        elif command == "undo_edit":
            return editor_tool.undo_edit(path)
        else:
            return f"Error: Unknown command '{command}'"
    return f"Error: Unknown tool '{tool_name}'"

def run_tools(message):
    tool_requests = [block for block in message.content if block.type == "tool_use"]
    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": tool_output,
                "is_error": False,
            }
        except Exception as e:
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True,
            }
        tool_result_blocks.append(tool_result_block)

    return tool_result_blocks

In [27]:
# Make the text edit schema based on the model version being used
def get_text_edit_schema(model):
  # Curso original: 3.7 → text_editor_20250124 + str_replace_editor
  if model.startswith("claude-3"):
    return {
      "type": "text_editor_20250124",
      "name": "str_replace_editor",
    }
  return {
    "type": "text_editor_20250728",
    "name": "str_replace_based_edit_tool",
  }

In [28]:
# Run the conversation in a loop until the model doesn't ask for a tool use
def run_conversation(messages):
    while True:
        response = chat(
            messages,
            tools=[get_text_edit_schema(model)],
        )

        add_assistant_message(messages, response)
        print(text_from_message(response))

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

    return messages

In [30]:
messages = []

add_user_message(
    messages,
    "Open the ./main.py and summarize its contents"
)

run_conversation(messages)

I'll open the main.py file and summarize its contents for you.
## Summary of main.py

This Python module calculates the mathematical constant **pi (π)** to a specified number of decimal digits. Here's what it contains:

### Main Components:

1. **`calculate_pi(digits=5)` function**: 
   - Implements the **Chudnovsky algorithm**, one of the fastest known algorithms for calculating pi
   - Converges rapidly at about 14 digits per iteration
   - Uses Python's `Decimal` module for high-precision arithmetic
   - Takes a `digits` parameter (default: 5) to specify decimal precision
   - Returns pi as a float rounded to the specified precision

2. **`get_pi_to_5th_digit()` function**:
   - A convenience wrapper that specifically calculates pi to 5 decimal places
   - Returns 3.14159

3. **Main execution block**:
   - When run as a script, it calculates pi to 5 decimal places
   - Prints the result in two formats:
     - Raw value
     - Formatted to 5 decimal places

The code is well-documente

[{'role': 'user', 'content': 'Open the ./main.py and summarize its contents'},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="I'll open the main.py file and summarize its contents for you.", type='text'),
   ToolUseBlock(id='toolu_011FkvHJ1RJQu3ooQjVE2YyD', caller=DirectCaller(type='direct'), input={'command': 'view', 'path': './main.py'}, name='str_replace_based_edit_tool', type='tool_use', toolset_name=None)]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_011FkvHJ1RJQu3ooQjVE2YyD',
    'content': '1: """\n2: Module to calculate pi to a specified number of digits.\n3: """\n4: \n5: def calculate_pi(digits=5):\n6:     """\n7:     Calculate pi to the specified number of decimal digits using the Chudnovsky algorithm.\n8:     \n9:     The Chudnovsky algorithm is one of the fastest known algorithms for calculating pi.\n10:     It converges very rapidly, gaining about 14 digits per iteration.\n11:     \n12:     Args:\n13:         dig

In [29]:
# Ejemplo de uso: crear un archivo main.py y sus pruebas unitarias
messages = []
add_user_message(
    messages,
    "Create a ./main.py file with a function to calculate pi to the 5th digit, and then create a ./test.py file to test it."
)
run_conversation(messages)

I'll create two files for you: a main.py file with a function to calculate pi to the 5th digit, and a test.py file to test it.
Let me check the existing main.py file and then create it:
Now I'll overwrite the existing main.py with the pi calculation function:
Perfect! I've created both files for you:

## **main.py**
This file contains:
- `calculate_pi(digits=5)`: A function that calculates pi to a specified number of decimal digits using the Chudnovsky algorithm (one of the fastest known algorithms for calculating pi)
- `get_pi_to_5th_digit()`: A convenience function that specifically returns pi to 5 decimal places (3.14159)
- A main block that demonstrates the function when run directly

## **test.py**
This file contains a comprehensive test suite with:
- `test_pi_to_5th_digit()`: Tests that pi is correctly calculated to 3.14159
- `test_pi_accuracy()`: Compares the result with Python's math.pi
- `test_pi_different_precisions()`: Tests the function with 2, 3, 5, and 10 decimal places
-

[{'role': 'user',
  'content': 'Create a ./main.py file with a function to calculate pi to the 5th digit, and then create a ./test.py file to test it.'},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="I'll create two files for you: a main.py file with a function to calculate pi to the 5th digit, and a test.py file to test it.", type='text'),
   ToolUseBlock(id='toolu_012UJtHrcACBvykGpmz5tNC1', caller=DirectCaller(type='direct'), input={'command': 'create', 'path': './main.py', 'file_text': '"""\nModule to calculate pi to a specified number of digits.\n"""\n\ndef calculate_pi(digits=5):\n    """\n    Calculate pi to the specified number of decimal digits using the Chudnovsky algorithm.\n    \n    The Chudnovsky algorithm is one of the fastest known algorithms for calculating pi.\n    It converges very rapidly, gaining about 14 digits per iteration.\n    \n    Args:\n        digits (int): Number of decimal digits to calculate (default: 5)\n    \n    Returns:\n      